In [ ]:
import datetime
import polars as pl
import hvplot
# import hvplot.pandas
# from hvplot.plotting import scatter_matrix
# import matplotlib.pyplot as plt

df_aurora = pl.read_parquet('data/level1/ne300_aurora.parquet')
df_aurora = df_aurora.with_columns(pl.col('dtm').cast(pl.Datetime(time_zone='UTC')))
# display(df_aurora.schema)

df_mkndaq = pl.read_parquet('data/level1/ne300_mkndaq.parquet')
# display(df_mkndaq.schema)

# set(df_aurora.columns + df_mkndaq.columns)

In [ ]:
df_cpd2 = pl.read_parquet('data/level1/2024/S11a.parquet')

MAPPINGS = pl.read_csv('cdp2_aurora_mappings.csv', has_header=True, dtypes=[pl.String]*4)
_mappings = MAPPINGS.filter(pl.col('cpd2_id').is_in(df_cpd2.columns))
mapping = dict(zip(_mappings['cpd2_id'], _mappings['aurora_id']))
df_cpd2 = df_cpd2.rename(mapping=mapping)
df_cpd2 = df_cpd2.rename({'1': 'dtm'})                    

# display(df_cpd2['F2_S11'].unique())
# F2_S11 indicates a nephelometer flag. In this data set, there are - inter alia - the following flags with their frequency
# {'60AB': 212, '60BB': 19, '0093': 39412, '0097': 3, '20AB': 298, '20BB': 22, '2093': 1}
# mapping = {'60AB': 5, '60BB': 6, '0090': 0, '0093': 0, '0097': 1, '20AB': 3, '20BB': 4, '2093': 2,
#             'C097':}
# df_cpd2 = df_cpd2.with_columns(pl.col('F2_S11').replace(mapping).cast(pl.Int8).alias('F2_S11_new'))
def foo(str) -> int:
    return int(str, 16)

df_cpd2 = df_cpd2.with_columns(pl.col('F2_S11').map_elements(foo).cast(pl.Int32).alias('F2_S11_new'))
# df_cpd2.plot(x='dtm', y='F2_S11_new')

In [ ]:
import numpy as np
# compute Angstrom exponents from the scattering data
def alpha(sigma: list[np.ndarray, np.ndarray], wavelength: list[int, int]) -> np.ndarray:
    """
    The Angstrom exponent (α) describes how aerosol scattering varies with wavelength. It can be calculated between two wavelengths as follows:
        α=-log(σscat(λ1)/σscat(λ2))/log(λ1/λ2)

    where:
        σscat(λ1) and σscat(λ2) are the scattering coefficients at two different wavelengths λ1 and λ2.
        α is the Angstrom exponent.

    Args:
        sigma (list[float, float, float]): sigma_scattering for the 3 wavelengths available.
        wavelength (list[int, int, int]): the 3 wavelengths available.

    Returns:
        list[float, float]: mean of Angstroem exponent and standard deviation
    """
    try:
        alpha = -np.log(sigma[0]/sigma[1]) / np.log(wavelength[0]/wavelength[1])
        return alpha
    except Exception as err:
        return None

def add_alpha_mean_sd(df):
    # Extract the scattering values as NumPy arrays
    sigma_450 = df['3450000'].to_numpy()
    sigma_525 = df['3525000'].to_numpy()
    sigma_635 = df['3635000'].to_numpy()

    # Calculate the Angstrom exponents for each pair of wavelengths using NumPy arrays
    alphas = np.array([
        alpha([sigma_635, sigma_525], [635, 525]),
        alpha([sigma_525, sigma_450], [525, 450]),
        alpha([sigma_450, sigma_635], [450, 635]),
    ])

    # Calculate the mean and standard deviation along the first axis (rows)
    angstrom_exponent = np.mean(alphas, axis=0)
    sd_angstrom_exponent = np.std(alphas, axis=0)

    # Add the mean and std as new columns to the DataFrame
    df = df.with_columns([
        pl.Series('angstrom_exponent', angstrom_exponent),
        pl.Series('sd_angstrom_exponent', sd_angstrom_exponent)
    ])
    return df

df_aurora = add_alpha_mean_sd(df_aurora)

df_cpd2 = add_alpha_mean_sd(df_cpd2)


In [ ]:
# visualize data of both instruments for a short period and with the different instrument flags reported
start = datetime.datetime(2024,7,20,0,0,0, tzinfo=datetime.timezone.utc)
# end = datetime.datetime(2024,7,16,0,0,0, tzinfo=datetime.timezone.utc)
end = start + datetime.timedelta(days=1)

df_ne300 = df_aurora.filter((pl.col('dtm').is_between(start, end)) & (pl.col('4035')==0))
fig_ne300 = df_ne300.plot.line(x='dtm', y=['3635000', '3635090', '3525000', '3525090', '3450000', '3450090'], 
                         title="NE300 (flag '4035'==0)")
fig_ne300_2 = df_ne300.plot.scatter(x='dtm', y=['operation', '4035'], 
                         title="NE300 (flag '4035'==0)")
fig_ae_ne300 = df_ne300.plot(x='dtm', y=['angstrom_exponent', 'sd_angstrom_exponent'])

df_ne300_3 = df_aurora.filter((pl.col('dtm').is_between(start, end)) & (pl.col('4035')!=0))
fig_ne300_3 = df_ne300_3.plot.scatter(x='dtm', y=['3635000', '3635090', '3525000', '3525090', '3450000', '3450090'], 
                         title="NE300 (flag '4035'!=0)")

df_3000 = df_cpd2.filter((pl.col('dtm').is_between(start, end)) & (pl.col('F2_S11_new')==147))

fig_3000 = df_3000.plot.line(x='dtm', y=['3635000', '3635090', '3525000', '3525090', '3450000', '3450090'],
                        title="Aurora3000 (flag 'F2_S11_new'==147)")
fig_3000_2 = df_3000.plot.scatter(x='dtm', y=['F2_S11_new'],
                        title="Aurora3000 (flag 'F2_S11_new'==147)")
fig_ae_3000 = df_cpd2.plot(x='dtm', y=['angstrom_exponent', 'sd_angstrom_exponent'])

df_3000_3 = df_cpd2.filter((pl.col('dtm').is_between(start, end)) & (pl.col('F2_S11_new')!=147))
fig_3000_3 = df_3000_3.plot.scatter(x='dtm', y=['3635000', '3635090', '3525000', '3525090', '3450000', '3450090'],
                        title=f"Aurora3000 (flag 'F2_S11_new'!=147)")

# hvplot.show(fig_ne300 + fig_ne300_3)
# hvplot.show(fig_3000 + fig_3000_3)
# hvplot.show(fig_ne300 + fig_3000)

# hvplot.show(fig_ae_ne300 + fig_ae_3000)


# fig_ne300_3000_525 = df_ne300_3000.plot.scatter(x='3000_3525000', y='ne300_3525000')
# fig_ne300_3000_450 = df_ne300_3000.plot.scatter(x='3000_3450000', y='ne300_3450000')
# df_1_1 = pl.DataFrame({'x': [0, 250], 'y': [0, 250]})
# fig_1_1 = df_1_1.plot.scatter(x='x', y='y')

# hvplot.show(fig_ne300_3000_635 + fig_ne300_3000_525 + fig_ne300_3000_450)

In [ ]:
import matplotlib.pyplot as plt

days = range(1, 13)
for day in days:
    # visualize data of both instruments for a short period and with the different instrument flags reported
    start = datetime.datetime(2024,8,day,0,0,0, tzinfo=datetime.timezone.utc)
    # end = datetime.datetime(2024,7,16,0,0,0, tzinfo=datetime.timezone.utc)
    end = start + datetime.timedelta(days=1)

    df_ne300 = df_aurora.filter((pl.col('dtm').is_between(start, end)) & (pl.col('4035')==0))
    fig_ne300 = df_ne300.plot.line(x='dtm', y=['3635000', '3635090', '3525000', '3525090', '3450000', '3450090'], 
                            title="NE300 (flag '4035'==0)")
    df_3000 = df_cpd2.filter((pl.col('dtm').is_between(start, end)) & (pl.col('F2_S11_new')==147))

    fig_3000 = df_3000.plot.line(x='dtm', y=['3635000', '3635090', '3525000', '3525090', '3450000', '3450090'],
                            title="Aurora3000 (flag 'F2_S11_new'==147)")

    df_ne300_3000 = pl.concat([df_ne300.select(['dtm', '3635000', '3525000', '3450000']).rename({'3635000': 'ne300_3635000', 
                                                                                                '3525000': 'ne300_3525000', 
                                                                                                '3450000': 'ne300_3450000'}),
                            df_3000.select(['dtm', '3635000', '3525000', '3450000']).rename({'3635000': '3000_3635000', 
                                                                                                '3525000': '3000_3525000', 
                                                                                                '3450000': '3000_3450000'})], how='align')
    fig_ne300_3000_635 = df_ne300_3000.plot.scatter(x='3000_3635000', y='ne300_3635000', title=f'{start} thru {end}')

    hvplot.show(fig_3000 + fig_ne300 + fig_ne300_3000_635, title=f'ne300_vs_3000-202407{day}')
    # hvplot.save(fig_3000 + fig_ne300 + fig_ne300_3000_635, filename=f'ne300_vs_3000-202407{day}.png')



